# Pilot 09 — Joint gain–offset envelope (step 6)

**Purpose.** Replace the SM's one-dimensional, single-branch gain sensitivity with a calibration
hypothesis propagated *consistently* through the three branches and maximised over the **joint**
$(g,b)$ envelope that the BME280 specification actually justifies.

**Status:** pilot / scouting. The envelope *assumption* is unchanged from the SM; only the
propagation is. Numbers are provisional until the block-length and held-out repairs (nb 01/06) land.

**This notebook is the authoritative producer of the step-6 snapshot.** The final section writes

- `joint_gain_offset_envelope.json` — the frozen snapshot read by notebook 11. Contract:
  `worst_case[ch].E_sampled / u_syst_rectangular / sm_single_branch / E_linearised`,
  `envelope[ch].g_max / b_at_g_max`, `channel_only[c][ch]`, plus `meta`, `self_check`,
  `baseline`, `reference_slices`, `pivot_recentring`, `significance_H`;
- `joint_gain_offset_envelope.md` — human-readable report, rendered by the same
  `write_report` as the CLI script;
- `joint_gain_offset_envelope_samples.csv` — every evaluated calibration.

`E_sampled` is the number quoted downstream; it is the maximum over **all** feasible
points (joint vertices + channel-only vertices + edges + random interior draws).

**Companion script.** `joint_gain_offset_envelope.py` is the CLI batch version
(`--subsample / --quick / --edge / --random / --refine / --u-stat`). It shares the primitives
imported below and writes the same JSON schema; its default local refinement can only raise
`E_sampled`, so record which producer wrote the snapshot (`meta.script`).

## 0. The feasible $(g,b)$ envelope

For each channel the centred calibration hypothesis is

$$X_{\rm read}-X_0 = g_X\,(X_{\rm true}-X_0)+b_X,
\qquad e_X(X)=(g_X-1)(X-X_0)+b_X$$

and the BME280 accuracy bound $|e_X(X)|\le\delta_X$ over the campaign range
$[X_{\min},X_{\max}]$ defines a **parallelogram** in $(g,b)$:

$$\gamma_{\max}=\frac{2\delta_X}{X_{\max}-X_{\min}},\qquad
(1,\pm\delta_X),\quad
(1-\gamma_{\max},\,\delta_X+\gamma_{\max}(X_{\min}-X_0)),\quad
(1+\gamma_{\max},\,\delta_X-\gamma_{\max}(X_{\max}-X_0))$$

$\gamma_{\max}=2\delta_X/\mathrm{span}$ reproduces the SM endpoint gains exactly
(H 30.3 %, T 11.7 %, P 4.7 %), so the envelope **assumption** is unchanged.
The uncentred hypothesis $X_{\rm read}=g_XX_{\rm true}$, i.e. $b=(g-1)X_0$, lies **outside**
this envelope at the quoted endpoint gains and is kept only as a reference.

In [9]:
import sys, json, time, numpy as np, pandas as pd
from itertools import product
from pathlib import Path
from datetime import datetime, timezone
sys.path.insert(0, '.')
from chain_propagation import Models, Calibration, calibrate_K, propagate, d_beta_identity
from joint_gain_offset_envelope import (Envelope, evaluate, linearised_jacobian, write_report,
                                        DELTA, PIVOT, COLUMN, IDX, CHANNELS)

T0 = time.time()
MODELS = '../models/'
DATA   = '../../data/processed/full_data.csv'
mo = Models(MODELS+'nist/refractive_index.py', MODELS+'mathar/Mathar2007.py')
df = pd.read_csv(DATA)

SUBSAMPLE = 1          # set to 1 for the production run
d = df.iloc[::SUBSAMPLE]
T, H, P, R, y = (d.temperature.values, d.humidity.values, d.pressure.values,
                 d.counts_ratio.values, d.n_1762.values)
print(f'rows: {len(d)} of {len(df)}')

rows: 145784 of 145784


In [10]:
ARCHIVED_DALPHA_H  = 4.3334e-09
ARCHIVED_RESID_RMS = 1.9304e-07

K, _ = calibrate_K(mo, T, H, P, R, y)
base = propagate(mo, T, H, P, R, K)
base_read  = {ch: float(base.d_beta_read[IDX[ch]]) for ch in CHANNELS}
base_alpha = {ch: float(base.beta_data[IDX[ch]]) for ch in CHANNELS}

print(f'd_alpha_H(read) = {base_read["H"]:+.4e}   '
      f'[archived full-data value: {ARCHIVED_DALPHA_H:+.4e}]')
print(f'offset-removed residual RMS = {base.resid_offset_removed_rms:.4e} (RI units)')
ident = d_beta_identity(mo, T, H, P, R, K, Calibration())
identity_max = float(np.abs(ident - base.d_beta).max())
print(f'max |identity - propagate| = {identity_max:.3e}')

rel_dev = abs(base_read['H'] - ARCHIVED_DALPHA_H) / abs(ARCHIVED_DALPHA_H)
SELF_CHECK = {
    'identity_max_abs'   : identity_max,
    'archived_checked'   : bool(SUBSAMPLE == 1),
    'd_alpha_H_rel_dev'  : float(rel_dev),
    'd_alpha_H_ok'       : bool(SUBSAMPLE == 1 and rel_dev < 1e-4),
    'archived_d_alpha_H' : ARCHIVED_DALPHA_H,
    'archived_resid_rms' : ARCHIVED_RESID_RMS,
    'resid_rms_rel_dev'  : float(abs(base.resid_offset_removed_rms - ARCHIVED_RESID_RMS)
                                 / ARCHIVED_RESID_RMS),
}
if SUBSAMPLE == 1:
    print(f'self-check: d_alpha_H(read) vs archived -> rel. dev {rel_dev:.2e}  '
          f'[{"PASS" if SELF_CHECK["d_alpha_H_ok"] else "CHECK"}]')
else:
    print('self-check: archived comparison skipped (subsample != 1)')
print('Gate: the baseline must reproduce the archived value, otherwise stop here.')

d_alpha_H(read) = +4.3334e-09   [archived full-data value: +4.3334e-09]
offset-removed residual RMS = 1.9304e-07 (RI units)
max |identity - propagate| = 8.152e-16
self-check: d_alpha_H(read) vs archived -> rel. dev 1.23e-06  [PASS]
Gate: the baseline must reproduce the archived value, otherwise stop here.


## 1. Envelope per channel

Span, worst-case gain deviation, and the size of the uncentred hypothesis relative to the
specification it is supposed to respect.

In [11]:
env = {ch: Envelope(ch, d[COLUMN[ch]].min(), d[COLUMN[ch]].max(), DELTA[ch], PIVOT[ch])
       for ch in CHANNELS}

env_tbl = pd.DataFrame([{
    'min': env[ch].xmin, 'max': env[ch].xmax, 'span': env[ch].span,
    'delta (spec)': DELTA[ch],
    'gamma_max = 2*delta/span': env[ch].gamma_max,
    'g_max': 1 + env[ch].gamma_max,
    'b at g_max': env[ch].vertices()[3][1],
    'uncentred b = (g-1)X0': env[ch].gamma_max * env[ch].x0,
    'uncentred / spec': env[ch].gamma_max * env[ch].x0 / DELTA[ch],
} for ch in CHANNELS], index=list(CHANNELS))
display(env_tbl.round(4))
print('The uncentred hypothesis leaves the stated specification: reference only.')

,min,max,span,delta (spec),gamma_max = 2*delta/span,g_max,b at g_max,uncentred b = (g-1)X0,uncentred / spec
T,17.5182,34.6225,17.1043,1.0,0.1169,1.1169,-0.1252,2.9232,2.9232
H,18.9356,45.3206,26.3850,4.0,0.3032,1.3032,-0.6452,9.0961,2.2740
P,964.6050,1006.7743,42.1693,1.0,0.0474,1.0474,-0.0327,46.7165,46.7165


The uncentred hypothesis leaves the stated specification: reference only.


## 2. Two envelopes: channel-only and joint

A single physical sensor has one gain error per channel, and all three channels feed the same
Ciddor anchor, so the humidity coefficient also moves when the **temperature** or **pressure**
gain is wrong. Two numbers are therefore reported for every response:

- **channel-only** — the four vertices of one channel, the others at the identity calibration;
  this is what the SM's single-branch endpoint is trying to estimate;
- **joint** — all $4^3$ vertices of the three-dimensional envelope.

For a linear response the triangle sum of the channel-only maxima would bound the joint maximum.
Whether it does is a test, not an assumption.

In [12]:
def theta_at(ch, g, b):
    t = {c: (1.0, 0.0) for c in CHANNELS}
    t[ch] = (g, b)
    return t

rows = []
for combo in product(*[env[ch].vertices() for ch in CHANNELS]):
    rows.append(evaluate(mo, T, H, P, R, K, base_read, 'joint_vertex',
                         dict(zip(CHANNELS, combo))))
for ch in CHANNELS:
    for g, b in env[ch].vertices():
        rows.append(evaluate(mo, T, H, P, R, K, base_read,
                             f'channel_only_{ch}', theta_at(ch, g, b)))
tbl = pd.DataFrame(rows)
print(f'evaluations: {len(tbl)} ({4**3} joint vertices + {4*3} channel-only)')

E_joint = {ch: float(tbl[tbl.kind == 'joint_vertex'][f'delta_{ch}'].abs().max())
           for ch in CHANNELS}
E_chan = {c: {r: float(tbl[tbl.kind == f'channel_only_{c}'][f'delta_{r}'].abs().max())
              for r in CHANNELS} for c in CHANNELS}
sm_single = {ch: abs(base_alpha[ch]) * env[ch].gamma_max for ch in CHANNELS}

comp = pd.DataFrame([{
    'from T': E_chan['T'][ch], 'from H': E_chan['H'][ch], 'from P': E_chan['P'][ch],
    'triangle sum': sum(E_chan[c][ch] for c in CHANNELS),
    'joint max': E_joint[ch],
    'SM single-branch': sm_single[ch],
    'SM / joint': sm_single[ch] / E_joint[ch],
} for ch in CHANNELS], index=[f'd_alpha_{ch}' for ch in CHANNELS])
display(comp.round(12))

evaluations: 76 (64 joint vertices + 12 channel-only)


,from T,from H,from P,triangle sum,joint max,SM single-branch,SM / joint
d_alpha_T,7.570000e-10,6.300000e-10,8.000000e-12,1.395000e-09,1.484000e-09,1.034520e-07,69.701535
d_alpha_H,2.930000e-10,6.590000e-10,0.000000e+00,9.520000e-10,1.024000e-09,3.988000e-09,3.894694
d_alpha_P,3.800000e-11,8.700000e-11,9.700000e-11,2.220000e-10,2.320000e-10,1.230700e-08,52.951454


### Sampling adequacy and linearity

The vertex maximum is a corner evaluation. To check that no interior point does better, sample the
four edges of each channel and draw uniform interior points; then compare with a first-order
estimate from the baseline Jacobian. If the linearised value falls below the sampled maximum, the
response is non-linear and the sampled maximum — not a first-order bound — is the number to quote.

In [13]:
rng = np.random.default_rng(42)
rows2 = []
for ch in CHANNELS:
    for g, b in env[ch].edge_samples(5):
        rows2.append(evaluate(mo, T, H, P, R, K, base_read, f'edge_{ch}', theta_at(ch, g, b)))
for _ in range(80):
    theta = {}
    for ch in CHANNELS:
        e = env[ch]
        g = rng.uniform(1 - e.gamma_max, 1 + e.gamma_max)
        lo, hi = e.b_bounds(g)
        theta[ch] = (g, rng.uniform(lo, hi))
    rows2.append(evaluate(mo, T, H, P, R, K, base_read, 'random', theta))
sweep = pd.DataFrame(rows2)
E_sweep = {ch: float(sweep[f'delta_{ch}'].abs().max()) for ch in CHANNELS}

# E_sampled = maximum over EVERY feasible point, matching the CLI script's
# feasible set: joint vertices + channel-only vertices + edges + random interior.
# A channel-only row puts the other channels at identity, which is inside the
# joint envelope, so it is a legal candidate for the maximum.
E_chan_max = {ch: max(E_chan[c][ch] for c in CHANNELS) for ch in CHANNELS}
E_sampled  = {ch: max(E_joint[ch], E_chan_max[ch], E_sweep[ch]) for ch in CHANNELS}

J = linearised_jacobian(mo, T, H, P, R, K, base_read, env)
bmax = {ch: env[ch].max_abs_b() for ch in CHANNELS}
E_lin = {r: float(sum(abs(J[c]['g'][r]) * env[c].gamma_max
                      + abs(J[c]['b'][r]) * bmax[c] for c in CHANNELS))
         for r in CHANNELS}

chk = pd.DataFrame([{
    'E vertices': E_joint[ch],
    'E channel-only': E_chan_max[ch],
    'E edges+random': E_sweep[ch],
    'E sampled': E_sampled[ch],
    'E linearised': E_lin[ch],
    'linearised < sampled?': E_lin[ch] < E_sampled[ch],
    'nonlinearity %': 100.0 * (E_sampled[ch] / E_lin[ch] - 1.0) if E_lin[ch] else np.nan,
    'joint > triangle sum?': E_joint[ch] > sum(E_chan[c][ch] for c in CHANNELS),
} for ch in CHANNELS], index=[f'd_alpha_{ch}' for ch in CHANNELS])
display(chk.round(12))
print('E_sampled is the maximum over the joint vertices, the channel-only rows and the '
      'edge/random sweep; quote E_sampled when linearised < sampled.')

,E vertices,E channel-only,E edges+random,E sampled,E linearised,linearised < sampled?,nonlinearity %,joint > triangle sum?
d_alpha_T,1.484000e-09,7.570000e-10,8.890000e-10,1.484000e-09,1.918000e-09,False,-22.623381,True
d_alpha_H,1.024000e-09,6.590000e-10,7.930000e-10,1.024000e-09,9.150000e-10,True,11.922982,True
d_alpha_P,2.320000e-10,9.700000e-11,1.790000e-10,2.320000e-10,2.250000e-10,True,3.110858,True


E_sampled is the maximum over the joint vertices, the channel-only rows and the edge/random sweep; quote E_sampled when linearised < sampled.


## 3. Comparison with the SM single-branch treatment

The SM endpoint is $(\delta g/g)_{\max}\,|\alpha_X|$ — it rescales **only** the fitted coefficient of
the same channel and omits the Ciddor-anchor and Mathar branches. The comparable
consistent-propagated number is the channel-only maximum; the joint maximum is larger.

In [14]:
u_stat   = 1.30e-9    # paired-bootstrap SE of the H difference (SM)
sm_u_syst = 2.30e-9    # SM systematic for the H coefficient
u_syst_rev = E_sampled['H'] / np.sqrt(3)
u_comb_sm  = float(np.hypot(u_stat, sm_u_syst))
u_comb_rev = float(np.hypot(u_stat, u_syst_rev))
dH = abs(base_read['H'])

sig = pd.DataFrame([
    {'u_syst': sm_u_syst, 'u_comb': u_comb_sm, 'significance (u_comb)': dH / u_comb_sm},
    {'u_syst': u_syst_rev, 'u_comb': u_comb_rev, 'significance (u_comb)': dH / u_comb_rev},
], index=['SM single-branch', 'consistent joint envelope'])
display(sig.round(12))
print(f'd_alpha_H = {dH:+.4e};  SM endpoint for the H coefficient = {sm_single["H"]:.4e}')
print(f'joint envelope max |delta_H| = {E_sampled["H"]:.4e}  '
      f'({sm_single["H"]/E_sampled["H"]:.1f}x smaller than the SM endpoint)')
print(f'H-only envelope max = {E_chan["H"]["H"]:.4e}  '
      f'({sm_single["H"]/E_chan["H"]["H"]:.1f}x smaller)')

,u_syst,u_comb,significance (u_comb)
SM single-branch,2.300000e-09,2.642000e-09,1.640218
consistent joint envelope,5.910000e-10,1.428000e-09,3.034385


d_alpha_H = +4.3334e-09;  SM endpoint for the H coefficient = 3.9878e-09
joint envelope max |delta_H| = 1.0239e-09  (3.9x smaller than the SM endpoint)
H-only envelope max = 6.5872e-10  (6.1x smaller)


## 4. Reference slices outside the envelope

The three hypotheses below are the ones a one-dimensional scan tends to use. Reporting them
alongside their envelope membership makes clear why they are not substitutes for the joint bound.

In [15]:
def inside(e, g, b):
    return all(abs((g - 1) * (x - e.x0) + b) <= e.delta + 1e-12 for x in (e.xmin, e.xmax))

REF_KINDS = (('uncentred_endpoint',  lambda e, gmax: (gmax - 1) * e.x0),
             ('centred_b0_endpoint', lambda e, gmax: 0.0),
             ('feasible_endpoint',   lambda e, gmax: e.vertices()[3][1]))

refs = []
for ch in CHANNELS:
    e = env[ch]
    gmax = 1 + e.gamma_max
    for kind, b_of in REF_KINDS:
        b = b_of(e, gmax)
        r = evaluate(mo, T, H, P, R, K, base_read, f'{kind}_{ch}', theta_at(ch, gmax, b))
        refs.append({'kind': kind, 'channel': ch, 'g': gmax, 'b': b,
                     'delta': r[f'delta_{ch}'], 'resid_rms': r['resid_rms'],
                     'inside envelope': inside(e, gmax, b)})
ref_tbl = pd.DataFrame(refs)
display(ref_tbl.round(12))

,kind,channel,g,b,delta,resid_rms,inside envelope
0,uncentred_endpoint,T,1.116929,2.923233,1.548000e-09,1.927960e-07,False
1,centred_b0_endpoint,T,1.116929,0.000000,6.240000e-10,1.927770e-07,False
2,feasible_endpoint,T,1.116929,-0.125158,5.760000e-10,1.927760e-07,True
3,uncentred_endpoint,H,1.303203,9.096079,-3.850000e-10,1.920730e-07,False
4,centred_b0_endpoint,H,1.303203,0.000000,-3.550000e-10,1.930460e-07,False
5,feasible_endpoint,H,1.303203,-0.645233,-3.530000e-10,1.931180e-07,True
6,uncentred_endpoint,P,1.047428,46.716468,-8.900000e-11,1.929410e-07,False
7,centred_b0_endpoint,P,1.047428,0.000000,-8.900000e-11,1.930320e-07,False
8,feasible_endpoint,P,1.047428,-0.032710,-8.900000e-11,1.930320e-07,True


## 5. What this establishes — and what it does not

**Establishes.** Under the same envelope assumption as the SM, consistent three-branch propagation
reduces the gain systematic by a factor of a few (not an order of magnitude), and the gain term
stops being the limiting systematic: the statistical / time-dependence term is larger.

**Does not establish.** A physical effect. A reduced gain systematic is not evidence for an
anomaly; it moves the attribution *away* from the humidity gain. The limiting constraints —
temporal dependence (nb 06) and Mathar domain coverage (91.7 % of rows above 25 °C) — still have to
be carried explicitly, and the block-length dependence of the paired interval (nb 07) is unchanged.

**Open convention.** The joint maximum is dominated by the temperature channel
($\gamma_T=11.7\%$ from $\pm1$ °C over a 17.1 K span). If that datasheet bound is judged too
conservative for a repeatedly used sensor, the H-only number is the fallback; state which one is
quoted.

In [16]:
print('HEADLINE — common (read) coordinate, full resolution')
print(f'  baseline d_alpha_H_read      = {base_read["H"]:+.4e}')
print(f'  SM single-branch endpoint    = {sm_single["H"]:.4e}')
print(f'  joint envelope max |delta_H| = {E_sampled["H"]:.4e}  '
      f'({sm_single["H"]/E_sampled["H"]:.1f}x smaller)')
print(f'  H-only envelope max          = {E_chan["H"]["H"]:.4e}  '
      f'({sm_single["H"]/E_chan["H"]["H"]:.1f}x smaller)')
print(f'  revised u_syst               = {u_syst_rev:.4e}  ->  u_comb {u_comb_rev:.4e}')
print(f'  significance                 = {dH/u_comb_rev:.2f} u_comb   (SM: {dH/u_comb_sm:.2f})')

HEADLINE — common (read) coordinate, full resolution
  baseline d_alpha_H_read      = +4.3334e-09
  SM single-branch endpoint    = 3.9878e-09
  joint envelope max |delta_H| = 1.0239e-09  (3.9x smaller)
  H-only envelope max          = 6.5872e-10  (6.1x smaller)
  revised u_syst               = 5.9116e-10  ->  u_comb 1.4281e-09
  significance                 = 3.03 u_comb   (SM: 1.64)


## 6. Export — the frozen step-6 snapshot

Writes the JSON consumed by notebook 11, the Markdown report (rendered by the same
`write_report` as the CLI script) and the full table of evaluated calibrations.
`E_sampled` is the number quoted everywhere downstream; the `self_check` block records
whether the full-resolution baseline reproduced the archived value and the identity check.

In [17]:
# ---------------------------------------------------------------------------
# 6. Export: frozen step-6 snapshot (consumed by notebook 11)
# ---------------------------------------------------------------------------
FEASIBLE_PREFIXES = ('joint_vertex', 'edge_', 'random', 'channel_only_')
feas = pd.concat([tbl, sweep], ignore_index=True)
feas = feas[feas['kind'].str.startswith(FEASIBLE_PREFIXES)].reset_index(drop=True)

# --- pivot re-centring invariance (same check as the CLI script) -----------
r0 = feas.loc[feas['delta_H'].abs().idxmax()]
theta_a = {ch: (float(r0[f'g_{ch}']), float(r0[f'b_{ch}'])) for ch in CHANNELS}
zero_pivot = {'T': 0.0, 'H': 0.0, 'P': 0.0}
theta_b = {ch: (theta_a[ch][0],
                theta_a[ch][1] + (theta_a[ch][0] - 1.0) * (zero_pivot[ch] - PIVOT[ch]))
           for ch in CHANNELS}
ra = evaluate(mo, T, H, P, R, K, base_read, 'pivot_check_a', theta_a)
rb = evaluate(mo, T, H, P, R, K, base_read, 'pivot_check_b', theta_b, pivots=zero_pivot)
PIVOT_CHECK = {k: float(abs(ra[k] - rb[k])) for k in ('resid_rms', 'resid_offset')}
for ch in CHANNELS:
    PIVOT_CHECK[f'd_alpha_{ch}_read'] = float(
        abs(ra[f'd_alpha_{ch}_read'] - rb[f'd_alpha_{ch}_read']))
print('pivot re-centring invariance (max abs deviation): '
      + '  '.join(f'{k}={v:.2e}' for k, v in PIVOT_CHECK.items()))

# --- summary in the CLI script's schema ------------------------------------
summary = {
    'meta': {
        'timestamp': datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
        'script': '09_joint_gain_offset_envelope.ipynb',
        'data': DATA, 'models': MODELS,
        'subsample': SUBSAMPLE, 'rows': int(len(d)), 'rows_total': int(len(df)),
        'quick': False, 'edge_points_per_edge': 5, 'random_draws': 80,
        'refine_draws_per_best': 0, 'linear_bound': True,
        'evaluations': int(len(feas)), 'evaluations_feasible': int(len(feas)),
        'channel_only_evaluations': 4 * len(CHANNELS),
        'seed': 42, 'delta': DELTA, 'pivot': PIVOT, 'u_comb': None,
        'elapsed_s': round(time.time() - T0, 1),
    },
    'self_check': SELF_CHECK,
    'envelope': {ch: {'min': float(env[ch].xmin), 'max': float(env[ch].xmax),
                      'span': float(env[ch].span),
                      'gamma_max': float(env[ch].gamma_max),
                      'g_max': float(1.0 + env[ch].gamma_max),
                      'max_abs_b': float(env[ch].max_abs_b()),
                      'b_at_g_max': float(env[ch].vertices()[3][1]),
                      'max_gain_at_zero_offset': float(env[ch].max_gain_at_zero_offset()),
                      'uncentred_b_at_g_max': float(env[ch].gamma_max * env[ch].x0),
                      'uncentred_exceeds_spec_by': float(
                          env[ch].gamma_max * env[ch].x0 / env[ch].delta)}
                 for ch in CHANNELS},
    'baseline': {'d_alpha_read': base_read, 'alpha_data': base_alpha,
                 'resid_rms': float(base.resid_offset_removed_rms),
                 'resid_offset': float(base.resid_offset), 'K': float(K)},
    'worst_case': {}, 'reference_slices': [], 'channel_only': {},
    'pivot_recentring': PIVOT_CHECK,
}

for ch in CHANNELS:
    col = f'delta_{ch}'
    rmax = feas.loc[feas[col].abs().idxmax()]
    sub_v = feas[feas['kind'] == 'joint_vertex']
    E_v = abs(float(sub_v.loc[sub_v[col].abs().idxmax()][col]))
    E_all = abs(float(rmax[col]))
    sm = float(sm_single[ch])
    summary['worst_case'][ch] = {
        'E_vertices': E_v, 'E_sampled': E_all, 'E_linearised': float(E_lin[ch]),
        'linearised_below_sampled': bool(E_lin[ch] < E_all),
        'u_syst_rectangular': float(E_all / np.sqrt(3.0)),
        'sm_single_branch': sm,
        'sm_over_consistent': float(sm / E_all) if E_all else None,
        'delta_min': float(feas[col].min()), 'delta_max': float(feas[col].max()),
        'attained_kind': str(rmax['kind']),
        'attained_calibration': {k: float(rmax[k])
                                 for k in ('g_T', 'b_T', 'g_H', 'b_H', 'g_P', 'b_P')},
        'attained_delta': float(rmax[col]),
        'attained_d_alpha_read': {c: float(rmax[f'd_alpha_{c}_read']) for c in CHANNELS},
        'resid_rms_at_max': float(rmax['resid_rms']),
    }

for c in CHANNELS:
    sub = feas[feas['kind'] == f'channel_only_{c}']
    summary['channel_only'][c] = {r: float(abs(sub[f'delta_{r}']).max()) for r in CHANNELS}

summary['reference_slices'] = [
    {'kind': r['kind'], 'channel': r['channel'], 'g': float(r['g']), 'b': float(r['b']),
     'delta': float(r['delta']), 'resid_rms': float(r['resid_rms'])} for r in refs]

u_stat_sm  = 1.30e-9     # SM nominal paired-bootstrap SE of the H difference
sm_u_syst  = 2.30e-9     # SM systematic for the H coefficient
d_h        = abs(base_read['H'])
u_syst_rev = summary['worst_case']['H']['u_syst_rectangular']
u_comb_sm  = float(np.hypot(u_stat_sm, sm_u_syst))
u_comb_rev = float(np.hypot(u_stat_sm, u_syst_rev))
summary['significance_H'] = {
    'd_alpha_H': d_h, 'u_stat': u_stat_sm,
    'sm_u_syst': sm_u_syst, 'sm_u_comb': u_comb_sm,
    'sm_significance': d_h / u_comb_sm,
    'revised_u_syst': float(u_syst_rev), 'revised_u_comb': u_comb_rev,
    'revised_significance': d_h / u_comb_rev,
    'sm_endpoint': float(sm_single['H'])}

OUT_JSON = 'joint_gain_offset_envelope.json'
OUT_MD   = 'joint_gain_offset_envelope.md'
OUT_CSV  = 'joint_gain_offset_envelope_samples.csv'
Path(OUT_JSON).write_text(json.dumps(summary, indent=2), encoding='utf-8')
feas.to_csv(OUT_CSV, index=False)
write_report(OUT_MD, summary)
print(f'\nwrote {OUT_JSON}\nwrote {OUT_CSV}\nwrote {OUT_MD}')
print(f'E_sampled (H) = {summary["worst_case"]["H"]["E_sampled"]:.4e}  '
      f'u_syst = {summary["worst_case"]["H"]["u_syst_rectangular"]:.4e}  '
      f'SM endpoint = {summary["worst_case"]["H"]["sm_single_branch"]:.4e}')

pivot re-centring invariance (max abs deviation): resid_rms=5.48e-21  resid_offset=4.45e-21  d_alpha_T_read=7.94e-21  d_alpha_H_read=1.46e-22  d_alpha_P_read=2.88e-19

wrote joint_gain_offset_envelope.json
wrote joint_gain_offset_envelope_samples.csv
wrote joint_gain_offset_envelope.md
E_sampled (H) = 1.0239e-09  u_syst = 5.9116e-10  SM endpoint = 3.9878e-09
